# AVA Spatial Agent Integration

Combines DroneKit control with GeoJSON spatial queries.
The drone can now find and navigate to conservation targets.

In [ ]:
!pip install -q agno dronekit pymavlink pydantic

In [ ]:
import os

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = input('OpenRouter API Key: ')

print('API key set.' if os.environ.get('OPENROUTER_API_KEY') else 'No API key!')

In [ ]:
from dronekit import connect, VehicleMode
import time

CONNECTION_STRING = 'tcp:127.0.0.1:5763'

print('Connecting to vehicle...')
vehicle = connect(CONNECTION_STRING, wait_ready=True, baud=57600, rate=60)
print(f'Connected!')
print(f'  Mode:     {vehicle.mode.name}')
print(f'  GPS:      {vehicle.gps_0}')
print(f'  Position: {vehicle.location.global_frame.lat}, {vehicle.location.global_frame.lon}')

## Import Spatial and Drone Tools

In [ ]:
from agno.tools import tool
from agents.spatial_agent import spatial_tools


@tool
def get_current_position() -> str:
    """
    Get current GPS coordinates of the drone.
    """
    lat = vehicle.location.global_frame.lat
    lon = vehicle.location.global_frame.lon
    alt = vehicle.location.global_relative_frame.alt
    return f"Lat: {lat}, Lon: {lon}, Alt: {alt}m"

@tool
def go_to_coords(lat: float, lon: float, alt: float = None) -> str:
    """
    Fly to GPS coordinates.
    Args:
        lat: Destination latitude
        lon: Destination longitude
        alt: Destination altitude in meters (optional)
    """
    from dronekit import LocationGlobalRelative
    
    if alt is None:
        alt = vehicle.location.global_relative_frame.alt
    
    target = LocationGlobalRelative(lat, lon, alt)
    vehicle.simple_goto(target)
    
    return f"Flying to {lat}, {lon} at {alt}m"

# Combine all tools
all_tools = spatial_tools + [get_current_position, go_to_coords]

print(f"Loaded {len(all_tools)} tools:")
for t in all_tools:
    print(f"  - {t.name}")

## Build Combined Agent

In [ ]:
from agno.agent import Agent  # Note: agno.agent not just agno
from agno.models.openrouter import OpenRouter

SYSTEM_PROMPT = """
You are AVA - an autonomous assistant for conservation drone operations.

You can:
- Query conservation targets from the GeoJSON database
- Find nearest targets to the drone's position
- Navigate the drone to target locations
- Provide target metadata and observation history

When asked to find or visit a target:
1. Get current drone position
2. Query for nearest target (or lookup by name)
3. Navigate to target coordinates
4. Report target details

Be concise and operational in tone.
"""

agent = Agent(
    name='AVA_Combined',
    model=OpenRouter(
        id='qwen/qwen3-vl-30b-a3b-thinking',  
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
    instructions=SYSTEM_PROMPT,
    tools=all_tools,
    add_history_to_context=True,
    debug_mode=True
)

print('Agent ready.')

## Test Queries

In [ ]:
# Test 1: Find nearest target to current position
result = agent.run("What's the nearest conservation target to my current position?")
print(result.content)

In [ ]:
# Test 2: Lookup specific target
result = agent.run("Tell me about Rocky Ridge Wildlife Habitat")
print(result.content)

In [ ]:
# Test 3: Navigate to nearest target
result = agent.run("Take me to the nearest high priority target")
print(result.content)

In [ ]:
# Test 4: List unvisited targets
result = agent.run("Show me all targets that haven't been visited yet")
print(result.content)

## Interactive Mode

In [ ]:
print('AVA Spatial Agent - Interactive Mode')
print('Type commands. Type "exit" to quit.\n')

while True:
    user_input = input('\nCommand: ')
    
    if user_input.lower().strip() == 'exit':
        break
    
    if not user_input.strip():
        continue
    
    try:
        result = agent.run(user_input)
        print(f'\nResponse: {result.content}')
    except Exception as e:
        print(f'\nERROR: {e}')

print('Session ended.')

In [ ]:
vehicle.close()
print('Vehicle connection closed.')